In [29]:
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.integrate import solve_ivp

In [30]:
# Parâmetros

N = 1000 # Número de espiras
g = 2e-3 # altura do gap
d = 4e-2 # largura do gap
L = 4e-2 # comprimento do gap
Lc = 70e-2 # comprimento do núcleo
m = 100e-3 # massa do êmbolo
R = 2 #resistor
k1 = 100 # constante elástica no primeiro problema
k2 = 1000 # constante elástica no segundo problema
x0 = 3e-2 # posição de repouso da mola
U0 = 4 * np.pi * 1e-7 # permeabilidade do vácuo
Ur = 71.6 # permeabilidade da aproximação linear

# Valores de BxH da tabela

H_tab = np.array([0, 68, 135, 203, 271, 338, 406, 474, 542, 609,
                  1100, 1500, 2500, 4000, 5000, 9000, 12000, 20000, 25000])
B_tab = np.array([0, 0.733, 1.205, 1.424, 1.517, 1.560, 1.588, 1.617,
                  1.631, 1.646, 1.689, 1.703, 1.724, 1.731, 1.738,
                  1.761, 1.770, 1.80, 1.816])

# Interpolação dado B, retorna H

H_interp = CubicSpline(B_tab, H_tab)

# Calculando a tensão da fonte
# Como está no regime permanente -> V = R * i_ss
B_ss = 1.8 # densidade de fluxo magnetico no regime permanente
H_ss = float(H_interp(B_ss)) # fluxo magnetico no regime permanente
i_ss = (H_ss * Lc + (B_ss * 2 * g) / U0) / N # corrente no regime permanente
V    = R * i_ss 

print(f"H(1.8T) = {H_ss:.1f} A/m")
print("Corrente no regime permanente:")
print(f"i_ss    = {i_ss:.3f} A")
print("Tensão da fonte:")
print(f"V       = {V:.3f} V")

H(1.8T) = 20000.0 A/m
Corrente no regime permanente:
i_ss    = 19.730 A
Tensão da fonte:
V       = 39.459 V


In [31]:
# Funções para calcular i(lamb,x) e F_mag(lamb,x)

# Função -> B_ferro e B_gap à partir de lambda e x

def B_ferro(lamb, x):
    return lamb / (N*d*L)

def B_gap(lamb, x):
    return lamb / (N*(d-x)*L)

# Função -> Corrente i(lambda, x) para os três casos

# H_fe(B_fe) depende da interpolação de B e H
def corrente_real(lamb, x):
    Bf = B_ferro(lamb, x)
    H_fe = float(H_interp(Bf))
    Bg = B_gap(lamb, x)
    return (H_fe*Lc + (Bg*2*g) / U0) / N 

# H_fe(B_fe) = B_fe / Ur * U0
def corrente_linear(lamb, x):
    Bf = B_ferro(lamb, x)
    H_fe = Bf / (Ur * U0)
    Bg   = B_gap(lamb, x)
    return (H_fe * Lc + (Bg * 2 * g) / U0) / N

# H_fe(B_fe) = 0
def corrente_ideal(lamb, x):
    Bg = B_gap(lamb, x)
    return (Bg * 2 * g) / (U0 * N)

# Função -> Força magnética F_mag(lambda, x)

def F_mag(lamb, x):
    return -(lamb**2 * g) / (U0 * N**2 * (d-x)**2 * L)

In [32]:
# Sistema de EDOs

def sistema(t, y, corrente_func, k):
    lamb, x, v = y # y é um vetor com 3 estamos do sistema: lamb = y[0], x = y[1], v = y[2]

    i = corrente_func(lamb, x)
    fmag = F_mag(lamb, x)
    fmola = k * (x - x0)

    dlamb_dt = V - R*i # V -> tensão da fonte
    dx_dt = v # v -> velocidade
    dv_dt = (fmag - fmola) / m

    return [dlamb_dt, dx_dt, dv_dt] # retorna o vetor com as três derivadas

# Condição de parada x = 0

def anteparo(t, y, corrente_func, k):
    return y[1] # quando x=0, a simulação para

anteparo.terminal = True # para a simulação ao atingir x=0
anteparo.direction = -1 # só detecta quando x está diminuindo (indo de positivo para zero)

# Condições iniciais das EDOs

y0    = [0.0, x0, 0.0]  # [lamb, x, v]
t_simulacao = (0, 10)   # intervalo de tempo da simulação 0 -> 10s
pontos_intervalo = np.linspace(0, 10, 100000)  # pontos de saída, 100000 pontos no intervalo de 0 -> 10s 

# Simulação para os 3 casos, usando k1 (mola 100 N/m)

# Caso real: usa a tabela BxH para o núcleo
solucao_real   = solve_ivp(sistema, t_simulacao, y0, t_eval=pontos_intervalo,
                           args=(corrente_real, k1), events=anteparo, max_step=1e-4)

# Caso linear: usa permeabilidade relativa constante Ur
solucao_linear = solve_ivp(sistema, t_simulacao, y0, t_eval=pontos_intervalo,
                           args=(corrente_linear, k1), events=anteparo, max_step=1e-4)

# Caso ideal: núcleo com μ→∞, só o gap contribui para a FMM
solucao_ideal  = solve_ivp(sistema, t_simulacao, y0, t_eval=pontos_intervalo,
                           args=(corrente_ideal, k1), events=anteparo, max_step=1e-4)

print(f"Tempo de colisão (real)   = {solucao_real.t[-1]:.4f} s")
print(f"Tempo de colisão (linear) = {solucao_linear.t[-1]:.4f} s")
print(f"Tempo de colisão (ideal)  = {solucao_ideal.t[-1]:.4f} s")



Tempo de colisão (real)   = 0.0203 s
Tempo de colisão (linear) = 0.0209 s
Tempo de colisão (ideal)  = 0.0203 s
